# La lunghezza della registrazione è un predittore della classe?

Esperimento **esplorativo** su UniMiB e UniZgFall: una `RandomForestClassifier` riceve come **unica feature** la lunghezza originale della registrazione, `n_samples` (numero di campioni prima del padding). Nessun segnale grezzo, nessuna feature ingegnerizzata, nessun altro metadato è usato come input.

Scopo: capire se la sola durata contiene già informazione sulle classi, prima di analizzare il contenuto dei segnali.

**Cosa questo notebook NON dimostra.**
- La cross-validation usata è **stratificata standard**: *non* garantisce l'indipendenza tra soggetti (in UniMiB lo stesso soggetto può comparire in training e test; in UniZgFall l'identificativo del soggetto non è disponibile). I risultati **non** misurano la generalizzazione a persone mai viste.
- Un risultato superiore al baseline indica che la lunghezza è informativa rispetto alle label *nel protocollo usato*. Non dimostra, da sola, che altri modelli sfruttino il padding, né che la lunghezza sia la causa delle predizioni.

Tutto avviene in memoria: nessun file viene scritto, nessun modello viene salvato, nessun risultato precedente viene toccato. Il notebook va eseguito con `fall_detection_ml/` come directory di lavoro (come gli altri notebook del progetto).

In [1]:
import sys
sys.dont_write_bytecode = True  # evita di scrivere file .pyc negli import

import importlib.util
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold

import config as cfg
import utils

HAS_PLOTLY = importlib.util.find_spec('plotly') is not None
if HAS_PLOTLY:
    import plotly.express as px
    import plotly.graph_objects as go
else:
    print('ATTENZIONE: Plotly non disponibile, i grafici saranno omessi (nessuna installazione eseguita).')

RANDOM_STATE = 89
N_SPLITS_MAX = 5
N_ESTIMATORS = 200
FEATURES = ['n_samples']  # unico predittore ammesso

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 60)
print('Parametri: RandomForest n_estimators =', N_ESTIMATORS, '| random_state =', RANDOM_STATE,
      '| class_weight = balanced | CV stratificata, shuffle=True, fold massimi =', N_SPLITS_MAX)
if cfg.SEED != RANDOM_STATE:
    print('Nota: config.SEED =', cfg.SEED, 'differisce da RANDOM_STATE usato qui')

Parametri: RandomForest n_estimators = 200 | random_state = 89 | class_weight = balanced | CV stratificata, shuffle=True, fold massimi = 5


## Sezione 1 — Preparazione dei dati

Per ogni dataset:

- `n_samples` e `label` vengono letti da `data/raw_<dataset>_meta.csv` (file generato dal notebook di preparazione dati; `n_samples` è la lunghezza originale prima del padding). La lunghezza **non** viene ricavata contando gli zeri dell'array padded.
- Le label e l'elenco dei file sono riletti con le funzioni esistenti (`utils.load_labels`, `utils.list_files`) e confrontati con i metadati: stesso numero di righe, stessi `record_id` **nello stesso ordine**, stesse label.
- Si controllano valori mancanti, lunghezze non positive o non intere, label fuori dalla mappatura di `config.py`.

Se l'associazione lunghezza ↔ label non può essere verificata con certezza (file mancanti, errori di lettura, incoerenze), **l'analisi di quel dataset viene interrotta** e il motivo è riportato.

Limite noto: la correttezza di `n_samples` nel file dei metadati è assunta (generato da `utils.build_dataset_files`); qui non vengono riletti i CSV originali per ricontarla.

In [2]:
def rel(path):
    return os.path.relpath(path, cfg.PROJECT_DIR).replace(os.sep, '/')


def load_length_table(ds):
    info = cfg.DATASETS[ds]
    names = info['class_names']
    meta_path = cfg.DATA_DIR / ('raw_' + ds + '_meta.csv')
    problems = []
    out = {'table': None, 'problems': problems, 'meta_path': rel(meta_path), 'labels_path': rel(info['labels_file'])}
    if not meta_path.is_file():
        problems.append('metadati non trovati: ' + rel(meta_path))
        return out
    try:
        meta = pd.read_csv(meta_path)
        labels = utils.load_labels(ds)
        files = utils.list_files(ds)
    except Exception as exc:
        problems.append('errore di caricamento (' + type(exc).__name__ + '): ' + str(exc)[:200])
        return out
    missing_cols = [c for c in ('record_id', 'label', 'n_samples') if c not in meta.columns]
    if missing_cols:
        problems.append('colonne mancanti nei metadati: ' + str(missing_cols))
        return out
    if not (len(meta) == len(labels) == len(files)):
        problems.append('numero di righe diverso: metadati=' + str(len(meta)) + ', label=' + str(len(labels)) + ', file=' + str(len(files)))
        return out
    if meta['record_id'].duplicated().any():
        problems.append('record_id duplicati nei metadati')
    if list(meta['record_id']) != list(files):
        bad = [i for i, (a, b) in enumerate(zip(meta['record_id'], files)) if a != b]
        problems.append('ordine/identificativi dei record non coincidono con utils.list_files (prima discordanza alla riga ' + str(bad[0]) + ')')
    if meta['n_samples'].isna().any() or meta['label'].isna().any():
        problems.append('valori mancanti: n_samples=' + str(int(meta['n_samples'].isna().sum())) + ', label=' + str(int(meta['label'].isna().sum())))
    else:
        if not np.array_equal(meta['label'].to_numpy(dtype=int), labels):
            problems.append('label dei metadati diverse da utils.load_labels')
        ns = meta['n_samples'].to_numpy(dtype=float)
        if np.any(ns <= 0) or np.any(ns != np.round(ns)):
            problems.append('n_samples non positivo o non intero')
        if labels.min() < 0 or labels.max() >= len(names):
            problems.append('label fuori dalla mappatura di config.py')
    if problems:
        return out
    lab = meta['label'].astype(int)
    out['table'] = pd.DataFrame({
        'record_id': meta['record_id'],
        'label': lab,
        'class_name': [names[i] for i in lab],
        'n_samples': meta['n_samples'].astype(int),
    })
    return out


LOADED = {ds: load_length_table(ds) for ds in cfg.DATASETS}
DATA = {ds: r['table'] for ds, r in LOADED.items() if r['table'] is not None}
print('Dataset utilizzabili:', list(DATA) or 'nessuno')

Dataset utilizzabili: ['UniZgFall', 'UniMiB']


In [3]:
for ds, r in LOADED.items():
    names = cfg.DATASETS[ds]['class_names']
    print('=' * 80)
    print(ds)
    print('  metadati :', r['meta_path'])
    print('  label    :', r['labels_path'])
    if r['table'] is None:
        print('  ANALISI INTERROTTA per questo dataset:')
        for p in r['problems']:
            print('   -', p)
        continue
    df = r['table']
    print('  campioni utilizzabili:', len(df), '| classi osservate:', df['label'].nunique(), 'su', len(names), 'nella mappatura')
    print('  controlli superati: stesso numero di righe, record_id identici e nello stesso ordine, label identiche, nessun valore mancante, n_samples intero e positivo')
    dist = df.groupby(['label', 'class_name']).size().rename('n_registrazioni').reset_index()
    dist['%'] = (100 * dist['n_registrazioni'] / len(df)).round(1)
    display(dist.set_index('label'))
    display(df['n_samples'].describe().round(1).to_frame('n_samples').T)

UniZgFall
  metadati : data/raw_UniZgFall_meta.csv
  label    : ../ml_accelerometers/y_UniZgFall.csv
  campioni utilizzabili: 468 | classi osservate: 10 su 10 nella mappatura
  controlli superati: stesso numero di righe, record_id identici e nello stesso ordine, label identiche, nessun valore mancante, n_samples intero e positivo


,class_name,n_registrazioni,%
label,,,
0,MW,68,14.5
1,MR,68,14.5
2,MJ,68,14.5
3,WD,13,2.8
4,WU,13,2.8
5,FF,34,7.3
6,FS,34,7.3
7,FB,34,7.3
8,LD,34,7.3


,count,mean,std,min,25%,50%,75%,max
n_samples,468.0,1697.6,803.0,388.0,1156.5,1525.0,2019.0,5146.0


UniMiB
  metadati : data/raw_UniMiB_meta.csv
  label    : ../ml_accelerometers/y_UniMiB_SHAR.csv
  campioni utilizzabili: 1980 | classi osservate: 17 su 17 nella mappatura
  controlli superati: stesso numero di righe, record_id identici e nello stesso ordine, label identiche, nessun valore mancante, n_samples intero e positivo


,class_name,n_registrazioni,%
label,,,
0,Walking,60,3.0
1,Running,60,3.0
2,GoingUpS,60,3.0
3,GoingDownS,60,3.0
4,Jumping,60,3.0
5,SittingDown,60,3.0
6,StandingUpFS,60,3.0
7,LyingDownFS,60,3.0
8,StandingUpFL,60,3.0


,count,mean,std,min,25%,50%,75%,max
n_samples,1980.0,374.7,380.4,85.0,151.8,206.0,446.5,1863.0


## Sezione 2 — Random Forest con la sola lunghezza

- Input: `X = n_samples.reshape(-1, 1)` (una sola colonna); target: label multiclass originale (`FALL_CLASSES` **non** viene usata qui).
- Modello: `RandomForestClassifier(n_estimators=200, random_state=89, class_weight='balanced')`.
- Validazione: `StratifiedKFold(n_splits=5, shuffle=True, random_state=89)`. Se la classe più piccola ha meno di 5 campioni, il numero di fold è ridotto al suo numero di campioni; se risulta inferiore a 2, la CV non viene eseguita e se ne spiega il motivo.
- Ogni registrazione è predetta **una sola volta**, da un modello addestrato senza di essa (predizioni *out-of-fold*); le metriche sono calcolate su queste predizioni aggregate. Non si valuta mai sul training set.
- Baseline (valutato negli stessi fold): `DummyClassifier(strategy='most_frequent')`, addestrato su `X[train]`, `y[train]` di ciascun fold, quindi predice la classe più frequente **nel training fold**. In caso di parità sceglie la label più piccola (comportamento deterministico di scikit-learn).

**Attenzione:** la CV è stratificata sulle classi, non raggruppata per soggetto. Registrazioni dello stesso soggetto possono finire sia in training sia in test.

In [4]:
def run_cv(df, names):
    y = df['label'].to_numpy(dtype=int)
    X = df[FEATURES].to_numpy(dtype=float)
    assert X.shape == (len(y), 1), X.shape
    counts = np.bincount(y)
    counts = counts[counts > 0]
    k = int(min(N_SPLITS_MAX, counts.min()))
    if k < 2:
        return {'ok': False, 'reason': 'la classe meno numerosa ha ' + str(int(counts.min())) + ' campione/i: servono almeno 2 campioni per classe per una CV stratificata'}
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    oof_rf = np.full(len(y), -1, dtype=int)
    oof_bl = np.full(len(y), -1, dtype=int)
    fold_of = np.zeros(len(y), dtype=int)
    splits, bl_const, rows = [], [], []
    for f, (tr, te) in enumerate(skf.split(X, y), start=1):
        rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, class_weight='balanced')
        rf.fit(X[tr], y[tr])
        oof_rf[te] = rf.predict(X[te])
        bl = DummyClassifier(strategy='most_frequent')
        bl.fit(X[tr], y[tr])
        oof_bl[te] = bl.predict(X[te])
        fold_of[te] = f
        splits.append((tr, te))
        c = int(bl.classes_[np.argmax(bl.class_prior_)])
        bl_const.append(c)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            rows.append({'fold': f, 'n_train': len(tr), 'n_test': len(te), 'classe_baseline': names[c],
                         'bal_acc_RF': round(balanced_accuracy_score(y[te], oof_rf[te]), 4),
                         'bal_acc_baseline': round(balanced_accuracy_score(y[te], oof_bl[te]), 4)})
    return {'ok': True, 'k': k, 'X_shape': X.shape, 'oof_rf': oof_rf, 'oof_bl': oof_bl, 'fold_of': fold_of,
            'splits': splits, 'bl_const': bl_const, 'fold_table': pd.DataFrame(rows).set_index('fold'),
            'min_class_count': int(counts.min())}


RESULTS = {}
for ds, df in DATA.items():
    res = run_cv(df, cfg.DATASETS[ds]['class_names'])
    RESULTS[ds] = res
    print('=' * 80)
    if not res['ok']:
        print(ds, '- CV non eseguibile:', res['reason'])
        continue
    print(ds, '- fold:', res['k'], '(classe meno numerosa:', res['min_class_count'], 'campioni)',
          '| predizioni out-of-fold per', len(df), 'registrazioni')
    if res['k'] < N_SPLITS_MAX:
        print('  Numero di fold ridotto da', N_SPLITS_MAX, 'a', res['k'], 'perché la classe meno numerosa ha pochi campioni')
    display(res['fold_table'])

UniZgFall - fold: 5 (classe meno numerosa: 13 campioni) | predizioni out-of-fold per 468 registrazioni


,n_train,n_test,classe_baseline,bal_acc_RF,bal_acc_baseline
fold,,,,,
1,374,94,OT,0.3092,0.1
2,374,94,OT,0.1443,0.1
3,374,94,OT,0.3407,0.1
4,375,93,OT,0.2826,0.1
5,375,93,OT,0.2769,0.1


UniMiB - fold: 5 (classe meno numerosa: 60 campioni) | predizioni out-of-fold per 1980 registrazioni


,n_train,n_test,classe_baseline,bal_acc_RF,bal_acc_baseline
fold,,,,,
1,1584,396,FallingForw,0.1993,0.0588
2,1584,396,FallingForw,0.2059,0.0588
3,1584,396,FallingForw,0.2288,0.0588
4,1584,396,FallingForw,0.2271,0.0588
5,1584,396,FallingForw,0.2190,0.0588


## Sezione 3 — Metriche della Random Forest (out-of-fold)

Accuracy, balanced accuracy, F1 macro e weighted; precision, recall e F1 per classe; matrice di confusione aggregata (righe = classe vera, colonne = classe predetta). Tutte le metriche derivano dalle predizioni out-of-fold. Le metriche del baseline sono nella Sezione 4.

In [5]:
def overall_metrics(y, p):
    return {'accuracy': accuracy_score(y, p), 'balanced_accuracy': balanced_accuracy_score(y, p),
            'f1_macro': f1_score(y, p, average='macro', zero_division=0),
            'f1_weighted': f1_score(y, p, average='weighted', zero_division=0)}


def per_class_table(y, p, names):
    labels = np.arange(len(names))
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=labels, zero_division=0)
    return pd.DataFrame({'classe': names, 'precision': pr, 'recall': rc, 'f1': f1, 'support': sup,
                         'n_predette': np.bincount(p, minlength=len(names))}, index=labels).rename_axis('label')


def top_confused(cm, names, n=5):
    rows = []
    for i in range(len(names)):
        tot = cm[i].sum()
        if tot == 0:
            continue
        for j in range(len(names)):
            if i != j and cm[i, j] > 0:
                rows.append({'classe vera': names[i], 'classe predetta': names[j], 'n': int(cm[i, j]),
                             '% della classe vera': round(100 * cm[i, j] / tot, 1)})
    cols = ['classe vera', 'classe predetta', 'n', '% della classe vera']
    return pd.DataFrame(rows, columns=cols).sort_values('n', ascending=False).head(n).reset_index(drop=True)


METRICS = {}
for ds, res in RESULTS.items():
    if not res['ok']:
        continue
    y = DATA[ds]['label'].to_numpy(dtype=int)
    names = cfg.DATASETS[ds]['class_names']
    labels = np.arange(len(names))
    METRICS[ds] = {
        'rf_overall': overall_metrics(y, res['oof_rf']), 'bl_overall': overall_metrics(y, res['oof_bl']),
        'rf_class': per_class_table(y, res['oof_rf'], names), 'bl_class': per_class_table(y, res['oof_bl'], names),
        'cm': confusion_matrix(y, res['oof_rf'], labels=labels), 'cm_bl': confusion_matrix(y, res['oof_bl'], labels=labels),
    }

for ds, m in METRICS.items():
    names = cfg.DATASETS[ds]['class_names']
    print('=' * 80)
    print(ds, '- RANDOM FOREST (solo n_samples), predizioni out-of-fold')
    display(pd.DataFrame([m['rf_overall']], index=['RandomForest']).round(4))
    print('Metriche per classe')
    display(m['rf_class'].round(4))
    print('Matrice di confusione aggregata (conteggi; righe = vera, colonne = predetta)')
    display(pd.DataFrame(m['cm'], index=names, columns=names))
    print('Coppie di classi piu confuse (fuori diagonale)')
    display(top_confused(m['cm'], names))
if not METRICS:
    print('Nessun risultato disponibile.')

UniZgFall - RANDOM FOREST (solo n_samples), predizioni out-of-fold


,accuracy,balanced_accuracy,f1_macro,f1_weighted
RandomForest,0.2906,0.2694,0.2726,0.3019


Metriche per classe


,classe,precision,recall,f1,support,n_predette
label,,,,,,
0,MW,0.5625,0.5294,0.5455,68,64
1,MR,0.2838,0.3088,0.2958,68,74
2,MJ,0.3235,0.3235,0.3235,68,68
3,WD,0.4286,0.4615,0.4444,13,14
4,WU,0.4167,0.3846,0.4000,13,12
5,FF,0.1026,0.1176,0.1096,34,39
6,FS,0.1389,0.1471,0.1429,34,36
7,FB,0.0000,0.0000,0.0000,34,40
8,LD,0.0600,0.0882,0.0714,34,50


Matrice di confusione aggregata (conteggi; righe = vera, colonne = predetta)


...

,classe,precision,recall,f1,support,n_predette
label,,,,,,
0,Walking,0.6557,0.6667,0.6612,60,61
1,Running,0.5645,0.5833,0.5738,60,62
2,GoingUpS,0.1273,0.1167,0.1217,60,55
3,GoingDownS,0.1356,0.1333,0.1345,60,59
4,Jumping,0.1967,0.2000,0.1983,60,61
5,SittingDown,0.1719,0.1833,0.1774,60,64
6,StandingUpFS,0.2063,0.2167,0.2114,60,63
7,LyingDownFS,0.2459,0.2500,0.2479,60,61
8,StandingUpFL,0.2037,0.1833,0.1930,60,54


Matrice di confusione aggregata (conteggi; righe = vera, colonne = predetta)


,Walking,Running,GoingUpS,GoingDownS,Jumping,SittingDown,StandingUpFS,LyingDownFS,StandingUpFL,FallingForw,FallingBack,FallingRight,FallingLeft,Syncope,FallingBackSC,FallingWithPS,HittingObstacle
Walking,40,18,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
Running,18,35,0,0,0,0,1,3,3,0,0,0,0,0,0,0,0
GoingUpS,0,0,7,13,10,6,6,6,11,0,0,0,0,0,1,0,0
GoingDownS,0,0,14,8,11,5,11,6,5,0,0,0,0,0,0,0,0
Jumping,0,0,8,10,12,9,7,7,3,1,1,1,0,0,0,1,0
SittingDown,0,0,4,6,7,11,17,10,5,0,0,0,0,0,0,0,0
StandingUpFS,0,1,9,10,5,16,13,1,5,0,0,0,0,0,0,0,0
LyingDownFS,2,3,5,6,4,11,4,15,10,0,0,0,0,0,0,0,0
StandingUpFL,1,5,7,6,8,6,4,12,11,0,0,0,0,0,0,0,0
FallingForw,0,0,0,0,3,0,0,0,0,21,16,68,18,13,17,14,10


Coppie di classi piu confuse (fuori diagonale)


,classe vera,classe predetta,n,% della classe vera
0,Syncope,FallingRight,74,41.1
1,FallingLeft,FallingRight,69,38.3
2,FallingForw,FallingRight,68,37.8
3,HittingObstacle,FallingRight,67,37.2
4,FallingBackSC,FallingRight,64,35.6


## Sezione 4 — Baseline: classe più frequente

Il baseline predice una costante: la classe più frequente nel **training fold** (mai quella del test fold). Per costruzione, un predittore costante ha recall 1 su una sola classe e 0 sulle altre, quindi la sua balanced accuracy vale 1/(numero di classi). Risultato del baseline e della Random Forest sono mostrati separatamente e poi confrontati.

In [6]:
for ds, m in METRICS.items():
    names = cfg.DATASETS[ds]['class_names']
    res = RESULTS[ds]
    print('=' * 80)
    print(ds, '- BASELINE (classe piu frequente nel training fold), predizioni out-of-fold')
    print('Classe predetta dal baseline in ciascun fold:', [names[c] for c in res['bl_const']])
    display(pd.DataFrame([m['bl_overall']], index=['Baseline']).round(4))
    bl_cls = m['bl_class']
    display(bl_cls[bl_cls['n_predette'] > 0].round(4))
    print('(le classi mai predette dal baseline hanno precision, recall e F1 pari a 0 e sono omesse dalla tabella)')
    comp = pd.DataFrame([m['rf_overall'], m['bl_overall']], index=['RandomForest', 'Baseline'])
    comp.loc['Differenza (RF - baseline)'] = comp.loc['RandomForest'] - comp.loc['Baseline']
    print('Confronto')
    display(comp.round(4))

UniZgFall - BASELINE (classe piu frequente nel training fold), predizioni out-of-fold
Classe predetta dal baseline in ciascun fold: ['OT', 'OT', 'OT', 'OT', 'OT']


,accuracy,balanced_accuracy,f1_macro,f1_weighted
Baseline,0.2179,0.1,0.0358,0.078


,classe,precision,recall,f1,support,n_predette
label,,,,,,
9,OT,0.2179,1.0,0.3579,102,468


(le classi mai predette dal baseline hanno precision, recall e F1 pari a 0 e sono omesse dalla tabella)
Confronto


,accuracy,balanced_accuracy,f1_macro,f1_weighted
RandomForest,0.2906,0.2694,0.2726,0.3019
Baseline,0.2179,0.1000,0.0358,0.0780
Differenza (RF - baseline),0.0726,0.1694,0.2368,0.2239


UniMiB - BASELINE (classe piu frequente nel training fold), predizioni out-of-fold
Classe predetta dal baseline in ciascun fold: ['FallingForw', 'FallingForw', 'FallingForw', 'FallingForw', 'FallingForw']


,accuracy,balanced_accuracy,f1_macro,f1_weighted
Baseline,0.0909,0.0588,0.0098,0.0152


,classe,precision,recall,f1,support,n_predette
label,,,,,,
9,FallingForw,0.0909,1.0,0.1667,180,1980


(le classi mai predette dal baseline hanno precision, recall e F1 pari a 0 e sono omesse dalla tabella)
Confronto


,accuracy,balanced_accuracy,f1_macro,f1_weighted
RandomForest,0.1803,0.2160,0.2119,0.1748
Baseline,0.0909,0.0588,0.0098,0.0152
Differenza (RF - baseline),0.0894,0.1572,0.2021,0.1596


## Sezione 5 — Grafici Plotly

Per ciascun dataset: (1) box plot di `n_samples` per classe, (2) istogramma delle lunghezze, (3) matrice di confusione della Random Forest (colore = % per riga della classe vera, numero = conteggio), (4) confronto delle metriche principali tra Random Forest e baseline. I grafici sono solo mostrati nel notebook, non salvati.

In [7]:
def show_plots(ds):
    df = DATA[ds]
    m = METRICS[ds]
    names = cfg.DATASETS[ds]['class_names']
    fig = px.box(df, x='class_name', y='n_samples', points='outliers', hover_data=['record_id'],
                 category_orders={'class_name': list(names)},
                 title=ds + ': lunghezza originale (n_samples) per classe',
                 labels={'class_name': 'classe', 'n_samples': 'n_samples'})
    fig.show()
    fig = px.histogram(df, x='n_samples', nbins=60, title=ds + ': distribuzione di n_samples',
                       labels={'n_samples': 'n_samples'})
    fig.update_layout(yaxis_title='n. registrazioni', bargap=0.05)
    fig.show()
    cm = m['cm']
    norm = 100 * cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig = go.Figure(go.Heatmap(
        z=norm, x=list(names), y=list(names), text=cm, texttemplate='%{text}', colorscale='Blues', zmin=0, zmax=100,
        colorbar=dict(title='% riga'),
        hovertemplate='vera=%{y}<br>predetta=%{x}<br>% della classe vera=%{z:.1f}<br>n=%{text}<extra></extra>'))
    fig.update_yaxes(autorange='reversed', title='classe vera')
    fig.update_xaxes(title='classe predetta', tickangle=60)
    fig.update_layout(title=ds + ': matrice di confusione Random Forest (out-of-fold)', height=620, width=820)
    fig.show()
    keys = ['accuracy', 'balanced_accuracy', 'f1_macro', 'f1_weighted']
    fig = go.Figure()
    fig.add_bar(name='RandomForest', x=keys, y=[m['rf_overall'][k] for k in keys])
    fig.add_bar(name='Baseline (classe piu frequente)', x=keys, y=[m['bl_overall'][k] for k in keys])
    fig.update_layout(barmode='group', title=ds + ': Random Forest vs baseline (out-of-fold)', yaxis_title='valore',
                      yaxis_range=[0, 1])
    fig.show()


if not HAS_PLOTLY:
    print('Plotly non disponibile: grafici omessi.')
elif not METRICS:
    print('Nessun risultato da mostrare.')
else:
    for ds in METRICS:
        show_plots(ds)

## Sezione 6 — Analisi binaria aggiuntiva: caduta vs ADL (facoltativa, separata)

L'esperimento multiclass sopra resta invariato. Qui, **senza riaddestrare nulla**, le stesse predizioni out-of-fold della Random Forest vengono riclassificate in binario con `FALL_CLASSES` di `config.py` (non modificata): classe positiva = caduta.

Il baseline binario è calcolato dentro ciascun training fold: predice la classe binaria (caduta/ADL) più frequente nel training fold, con gli stessi fold della CV multiclass. In caso di parità sceglie ADL.

Avvertenza: per UniZgFall la mappatura `FALL_CLASSES = [5, 6, 7]` è un'impostazione del progetto **non verificata** (README di `fall_detection_ml`); le metriche binarie dipendono da quella scelta. Inoltre la RF è addestrata sul problema multiclass, non direttamente sul binario.

In [8]:
def binary_metrics(yb, p):
    tn, fp, fn, tp = confusion_matrix(yb, p, labels=[0, 1]).ravel()
    return {'precision': precision_score(yb, p, pos_label=1, zero_division=0),
            'recall': recall_score(yb, p, pos_label=1, zero_division=0),
            'f1': f1_score(yb, p, pos_label=1, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(yb, p),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)}


BIN = {}
for ds, res in RESULTS.items():
    if not res['ok']:
        continue
    fall = cfg.FALL_CLASSES.get(ds)
    names = cfg.DATASETS[ds]['class_names']
    print('=' * 80)
    if not fall or any(i < 0 or i >= len(names) for i in fall):
        print(ds, '- FALL_CLASSES non definita o non coerente con la mappatura: analisi binaria omessa')
        continue
    y = DATA[ds]['label'].to_numpy(dtype=int)
    yb = np.isin(y, fall).astype(int)
    p_rf = np.isin(res['oof_rf'], fall).astype(int)
    p_bl = np.full(len(y), -1, dtype=int)
    consts = []
    for tr, te in res['splits']:
        c = int(np.bincount(yb[tr], minlength=2).argmax())
        consts.append(c)
        p_bl[te] = c
    assert (p_bl >= 0).all()
    BIN[ds] = {'rf': binary_metrics(yb, p_rf), 'bl': binary_metrics(yb, p_bl), 'consts': consts}
    print(ds, '- classi di caduta (FALL_CLASSES):', [str(i) + '=' + names[i] for i in fall])
    print('  registrazioni di caduta:', int(yb.sum()), 'su', len(yb), '(', round(100 * yb.mean(), 1), '%)')
    print('  classe binaria predetta dal baseline per fold (1 = caduta, 0 = ADL):', consts)
    display(pd.DataFrame([BIN[ds]['rf'], BIN[ds]['bl']], index=['RandomForest (da multiclass)', 'Baseline binario']).round(4))
if not BIN:
    print('Nessuna analisi binaria disponibile.')

UniZgFall - classi di caduta (FALL_CLASSES): ['5=FF', '6=FS', '7=FB']
  registrazioni di caduta: 102 su 468 ( 21.8 %)
  classe binaria predetta dal baseline per fold (1 = caduta, 0 = ADL): [0, 0, 0, 0, 0]


,precision,recall,f1,balanced_accuracy,TP,FP,FN,TN
RandomForest (da multiclass),0.3043,0.3431,0.3226,0.5623,35,80,67,286
Baseline binario,0.0000,0.0000,0.0000,0.5000,0,0,102,366


UniMiB - classi di caduta (FALL_CLASSES): ['9=FallingForw', '10=FallingBack', '11=FallingRight', '12=FallingLeft', '13=Syncope', '14=FallingBackSC', '15=FallingWithPS', '16=HittingObstacle']
  registrazioni di caduta: 1440 su 1980 ( 72.7 %)
  classe binaria predetta dal baseline per fold (1 = caduta, 0 = ADL): [1, 1, 1, 1, 1]


,precision,recall,f1,balanced_accuracy,TP,FP,FN,TN
RandomForest (da multiclass),0.9965,0.9965,0.9965,0.9936,1435,5,5,535
Baseline binario,0.7273,1.0000,0.8421,0.5000,1440,540,0,0


## Sezione 7 — Conclusioni automatiche

Il testo seguente è generato dai valori effettivamente calcolati sopra. Non vengono usate soglie per giudicare se un risultato sia «alto»: si riportano le metriche osservate e il confronto con il baseline.

In [9]:
def rel_text(d):
    if d > 0:
        return 'supera'
    return 'è uguale a' if d == 0 else 'è inferiore a'


if not METRICS:
    print('Nessun risultato disponibile: nessuna conclusione possibile.')
for ds, m in METRICS.items():
    names = cfg.DATASETS[ds]['class_names']
    rf, bl = m['rf_overall'], m['bl_overall']
    d_b = rf['balanced_accuracy'] - bl['balanced_accuracy']
    d_f = rf['f1_macro'] - bl['f1_macro']
    print('=' * 80)
    print(ds)
    print('  [OSSERVATO] Balanced accuracy: RF', round(rf['balanced_accuracy'], 3), 'vs baseline', round(bl['balanced_accuracy'], 3),
          '(differenza', format(d_b, '+.3f') + ') -> la RF', rel_text(d_b), 'il baseline.')
    print('  [OSSERVATO] F1 macro: RF', round(rf['f1_macro'], 3), 'vs baseline', round(bl['f1_macro'], 3),
          '(differenza', format(d_f, '+.3f') + ') -> la RF', rel_text(d_f), 'il baseline.')
    print('  [OSSERVATO] Accuracy: RF', round(rf['accuracy'], 3), 'vs baseline', round(bl['accuracy'], 3),
          '| F1 weighted: RF', round(rf['f1_weighted'], 3), 'vs baseline', round(bl['f1_weighted'], 3))
    cls = m['rf_class']
    zero = cls.loc[cls['recall'] == 0, 'classe'].tolist()
    best = cls.sort_values('recall', ascending=False).head(3)
    print('  [OSSERVATO] Classi con recall RF = 0:', len(zero), 'su', len(names), zero if zero else '')
    print('  [OSSERVATO] Classi con recall RF piu alta:', [(r['classe'], round(r['recall'], 2)) for _, r in best.iterrows()])
    tc = top_confused(m['cm'], names, 3)
    print('  [OSSERVATO] Coppie piu confuse (vera -> predetta, n):',
          [(r['classe vera'] + ' -> ' + r['classe predetta'], int(r['n'])) for _, r in tc.iterrows()])
    if d_b > 0 or d_f > 0:
        print('  [INTERPRETAZIONE] Nel protocollo usato la sola lunghezza risulta informativa rispetto alle label (almeno una delle due metriche supera il baseline); l entità va letta nelle metriche sopra, non da una soglia.')
    else:
        print('  [INTERPRETAZIONE] Nel protocollo usato la sola lunghezza non migliora il baseline sulle metriche considerate.')
    if ds in BIN:
        b = BIN[ds]
        print('  [OSSERVATO] Binario caduta vs ADL: balanced accuracy RF', round(b['rf']['balanced_accuracy'], 3), 'vs baseline', round(b['bl']['balanced_accuracy'], 3),
              '| F1 (caduta) RF', round(b['rf']['f1'], 3), 'vs baseline', round(b['bl']['f1'], 3))
print()
print('LIMITI (valgono per tutti i risultati sopra)')
print(' - La CV stratificata non separa i soggetti: il risultato non misura la generalizzazione a persone mai viste (UniMiB: soggetti presenti in piu fold; UniZgFall: soggetti non identificabili).')
print(' - Un risultato superiore al baseline mostra che la lunghezza e informativa rispetto alle label in questo protocollo; non prova che altri modelli sfruttino il padding, ne che la lunghezza causi le predizioni.')
print(' - La lunghezza puo riflettere il protocollo di acquisizione (durata fissata o segmentazione per attivita) e non una proprieta del movimento: il notebook non permette di distinguere queste ipotesi.')
print(' - Frequenza di campionamento e unita non sono note: n_samples e un conteggio di campioni, non una durata in secondi.')
print(' - Con una sola feature discreta, molte registrazioni condividono la stessa lunghezza e classi diverse possono avere lunghezze sovrapposte: la RF non puo separarle.')
print(' - Un solo schema di CV con un solo seed: la variabilita rispetto alla scelta dei fold non e stata stimata (vedi le balanced accuracy per fold in Sezione 2).')
print(' - n_samples e preso dai metadati generati dal progetto; non e stato ricontrollato rileggendo i CSV originali.')

UniZgFall
  [OSSERVATO] Balanced accuracy: RF 0.269 vs baseline 0.1 (differenza +0.169) -> la RF supera il baseline.
  [OSSERVATO] F1 macro: RF 0.273 vs baseline 0.036 (differenza +0.237) -> la RF supera il baseline.
  [OSSERVATO] Accuracy: RF 0.291 vs baseline 0.218 | F1 weighted: RF 0.302 vs baseline 0.078
  [OSSERVATO] Classi con recall RF = 0: 1 su 10 ['FB']
  [OSSERVATO] Classi con recall RF piu alta: [('MW', 0.53), ('WD', 0.46), ('WU', 0.38)]
  [OSSERVATO] Coppie piu confuse (vera -> predetta, n): [('OT -> MJ', 25), ('MJ -> OT', 17), ('OT -> LD', 17)]
  [INTERPRETAZIONE] Nel protocollo usato la sola lunghezza risulta informativa rispetto alle label (almeno una delle due metriche supera il baseline); l entità va letta nelle metriche sopra, non da una soglia.
  [OSSERVATO] Binario caduta vs ADL: balanced accuracy RF 0.562 vs baseline 0.5 | F1 (caduta) RF 0.323 vs baseline 0.0
UniMiB
  [OSSERVATO] Balanced accuracy: RF 0.216 vs baseline 0.059 (differenza +0.157) -> la RF supera il b

## Controllo finale (eseguito nel notebook)

Verifiche automatiche su quanto effettivamente calcolato: coerenza metadati/label, unica feature, copertura delle predizioni out-of-fold, separazione train/test e origine della classe del baseline.

In [10]:
rows = []
for ds, res in RESULTS.items():
    if not res['ok']:
        rows.append({'dataset': ds, 'controllo': 'CV eseguita', 'esito': False})
        continue
    y = DATA[ds]['label'].to_numpy(dtype=int)
    n = len(y)
    tested = np.concatenate([te for _, te in res['splits']])
    disjoint = all(len(np.intersect1d(tr, te)) == 0 for tr, te in res['splits'])
    bl_ok = all(c == int(np.bincount(y[tr]).argmax()) for (tr, _), c in zip(res['splits'], res['bl_const']))
    bl_const_per_fold = all(len(np.unique(res['oof_bl'][te])) == 1 for _, te in res['splits'])
    checks = [
        ('metadati, record_id, ordine e label coerenti (nessun problema segnalato)', len(LOADED[ds]['problems']) == 0),
        ('unica feature = n_samples (X con una sola colonna)', FEATURES == ['n_samples'] and res['X_shape'] == (n, 1)),
        ('ogni registrazione e nel test di esattamente un fold', len(tested) == n and len(np.unique(tested)) == n),
        ('predizioni RF e baseline tutte presenti (out-of-fold)', bool((res['oof_rf'] >= 0).all() and (res['oof_bl'] >= 0).all())),
        ('training e test disgiunti in ogni fold', disjoint),
        ('classe del baseline = classe piu frequente del solo training fold', bl_ok and bl_const_per_fold),
        ('task multiclass (nessuna binarizzazione nel modello)', set(np.unique(y)).issubset(set(range(len(cfg.DATASETS[ds]['class_names']))))),
    ]
    for text, ok in checks:
        rows.append({'dataset': ds, 'controllo': text, 'esito': bool(ok)})
for ds, r in LOADED.items():
    if r['table'] is None:
        rows.append({'dataset': ds, 'controllo': 'dataset interrotto: ' + '; '.join(r['problems'])[:150], 'esito': False})
CHECKS = pd.DataFrame(rows)
display(CHECKS)
print('Tutti i controlli superati:', bool(CHECKS['esito'].all()) if len(CHECKS) else False)

,dataset,controllo,esito
0,UniZgFall,"metadati, record_id, ordine e label coerenti (...",True
1,UniZgFall,unica feature = n_samples (X con una sola colo...,True
2,UniZgFall,ogni registrazione e nel test di esattamente u...,True
3,UniZgFall,predizioni RF e baseline tutte presenti (out-o...,True
4,UniZgFall,training e test disgiunti in ogni fold,True
5,UniZgFall,classe del baseline = classe piu frequente del...,True
6,UniZgFall,task multiclass (nessuna binarizzazione nel mo...,True
7,UniMiB,"metadati, record_id, ordine e label coerenti (...",True
8,UniMiB,unica feature = n_samples (X con una sola colo...,True
9,UniMiB,ogni registrazione e nel test di esattamente u...,True


Tutti i controlli superati: True
